# S14 · Is this series steady enough to model?

One picture carries this whole notebook: a **price wanders off** and never
settles, while its **day-to-day returns sit still** around zero. Before anyone
forecasts or models a series, they check it is steady enough to bother. We look
at both pictures, then run a single plain check that gives a straight yes/no.

The formal name for "steady enough" is **stationary**, and the formal test and
the correlogram plots behind it are waiting in the optional **Stretch** cells
for anyone heading to ARIMA next session.

**New here? Read this once.**

- New to Python? Run each cell in order and read the note above it. No typing
  needed.
- New to the ideas? "Steady enough to model" (the technical word is
  *stationary*) just means the series keeps the same character over time: the
  same average and the same spread from start to finish. We show you the picture
  and hand you a one-line check.
- Already know time series? The **Stretch (optional)** cells open up the ADF
  hypothesis test behind the check and the ACF/PACF plots you'll want for ARIMA.
- Any unfamiliar word is in `primers/glossary.md`.

## Setup

If you are on **Google Colab**, run the next cell once. On your **own machine**
everything is already installed, so it does nothing there.

In [ ]:
# This notebook uses numpy, pandas and matplotlib (Colab has them) plus
# yfinance (for the live download) and statsmodels (which powers the one-line
# check below and the Stretch plots). Colab lacks yfinance; install both there.
import sys
if "google.colab" in sys.modules:
    !pip install -q yfinance statsmodels
else:
    print("Not on Colab - assuming the libraries are already installed.")

In [ ]:
import numpy as np                  # fast maths on lists of numbers
import pandas as pd                   # tables of data, indexed by date
import matplotlib.pyplot as plt       # drawing charts

# Same seed as the earlier notebooks so the offline fallback is identical.
np.random.seed(0)

## Step 1 — get the same data, and its log-returns

We reuse the same recipe one last time: try the live Nifty 50 download, fall
back to the seeded synthetic series offline, then compute log-returns exactly as
in notebook 2.

In [ ]:
# Try live Nifty 50 data; fall back to a seeded synthetic price series offline.
ticker_symbol = "^NSEI"
close_price = None
data_source = ""

try:
    import yfinance as yf
    downloaded = yf.download(
        ticker_symbol,
        start="2021-01-01",
        end="2025-01-01",
        auto_adjust=True,
        progress=False,
    )
    if downloaded is None or len(downloaded) == 0:
        raise ValueError("no data returned")
    close_price = downloaded["Close"].dropna().squeeze()
    data_source = "LIVE download from yfinance (" + ticker_symbol + ")"

except Exception:
    # OFFLINE FALLBACK: build a realistic geometric random walk.
    print("Could not download live data; using a synthetic price series instead.")
    number_of_days = 1000
    daily_drift = 0.0003
    daily_volatility = 0.012
    daily_shocks = np.random.normal(daily_drift, daily_volatility, size=number_of_days)
    starting_price = 15000.0
    price_levels = starting_price * np.exp(np.cumsum(daily_shocks))
    dates = pd.bdate_range(start="2021-01-01", periods=number_of_days)
    close_price = pd.Series(price_levels, index=dates, name="Close")
    data_source = "SYNTHETIC fallback (seeded geometric random walk)"

print("Data source:", data_source)
print("Number of days:", len(close_price))

## Step 2 — compute the log-returns

The same one line as notebook 2: the natural log of the price ratio from one day
to the next, with the first (empty) day dropped.

In [ ]:
# Natural log of (today / yesterday). Drop the first NaN.
log_returns = np.log(close_price / close_price.shift(1)).dropna()

print("Number of returns:", len(log_returns))

## Step 3 — the picture: a price wanders, its returns sit still

Here is the whole idea in one figure. On top we draw the **price**; below it we
draw the same series' **daily returns**, sharing the same dates.

- The **price wanders off**. It drifts up and down with no fixed level it comes
  back to, so its average keeps moving. There is nothing steady to grab onto.
- The **returns sit still** around a flat line at zero, with a roughly constant
  spread. Same character at the start, the middle and the end.

That contrast is the reason the whole finance block models *returns*, not
*prices*. A wandering line can't teach us about tomorrow; a steady one can.

In [ ]:
# Two panels sharing the dates: the wandering price on top, the steady
# returns below. Look at the contrast, not the exact numbers.
figure, (top, bottom) = plt.subplots(
    2, 1, figsize=(9, 7), sharex=True
)

# Top panel: the price wanders off and never settles.
top.plot(close_price.index, close_price.values, color="#2E75B6")
top.set_ylabel("closing price")
top.set_title("A price wanders off (no fixed level)")

# Bottom panel: the day-to-day returns sit still around zero.
bottom.plot(log_returns.index, log_returns.values * 100,
            color="#C0392B", linewidth=0.8)
bottom.axhline(0, color="grey", linewidth=1)
bottom.set_ylabel("daily return (%)")
bottom.set_xlabel("date")
bottom.set_title("Its day-to-day returns sit still around zero")

plt.tight_layout()
plt.show()

## Step 4 — the same thing, in numbers

A quick way to feel the difference without any statistics: cut each series in
half and compare the two halves' averages. A **steady** series should have
similar averages in both halves; a **wandering** one won't.

In [ ]:
# Split each series in half and compare averages.
half = len(close_price) // 2
print("PRICE   first-half average :", round(float(close_price.iloc[:half].mean()), 2))
print("PRICE   second-half average:", round(float(close_price.iloc[half:].mean()), 2))
print("        (these usually differ a lot -> the price drifts -> not steady)")
print()

half_r = len(log_returns) // 2
print("RETURNS first-half average :", round(float(log_returns.iloc[:half_r].mean()), 5))
print("RETURNS second-half average:", round(float(log_returns.iloc[half_r:].mean()), 5))
print("        (both sit near zero -> the returns are steady)")

## Step 5 — one plain check: `is_it_steady`

Eyeballing plots and half-averages is a good start, but before you model a
series you want a straight yes/no. Here is a small helper, `is_it_steady`, that
gives exactly that. Hand it any series and it prints a plain verdict:

```
steady enough to model: yes
```

Under the hood it runs a standard statistical test from `statsmodels` and reads
off its summary number, but you don't need any of that machinery today. The
function is a **black box**: numbers in, plain verdict out. (Curious what's
inside? The first Stretch cell opens the box and names the test.)

In [ ]:
from statsmodels.tsa.stattools import adfuller


def is_it_steady(series, name="series"):
    """Plain yes/no on whether a series is steady enough to model.

    Hands the numbers to a standard statistical test and turns its result into
    one plain line. You do not need to know how the test works to use this;
    that lives in the Stretch cell below.
    """
    values = np.asarray(series, dtype=float)
    summary_number = adfuller(values)[1]     # the test's one summary number
    steady = summary_number < 0.05
    answer = "yes" if steady else "no"
    print(name, "-> steady enough to model:", answer)
    return steady

## Step 6 — run the check on the price, then on the returns

Now the payoff. We run the exact same check on the wandering price and on its
day-to-day returns. Watch the verdict **flip**: the price is *not* steady enough
to model, but its returns are. That flip is the whole reason we switch from
price to return before doing anything else.

In [ ]:
# Same check, both series. The verdict should flip from no to yes.
# (We keep the returned yes/no in a variable so nothing extra is printed.)
price_is_steady = is_it_steady(close_price, name="PRICE  ")
returns_are_steady = is_it_steady(log_returns, name="RETURNS")

And there it is. **Price: no. Returns: yes.** You now have the finance block's
core move in one line of code: take any price series, turn it into returns, and
the `is_it_steady` check turns green. That is the green light to model it, which
is exactly what next session does.

If you're happy with that, you're done with the main path. The Stretch cells
below open the black box and show the two plots you'll actually use in ARIMA.

### Stretch (optional) — what's inside `is_it_steady`: the ADF test

Skip this if you're new. The check above wraps the **ADF test** (Augmented
Dickey-Fuller), the standard hypothesis test for stationarity. Read it like any
hypothesis test:

- **Null hypothesis:** the series is *non-stationary* (it has a *unit root* — a
  random-walk trend).
- The test returns a **p-value**. A **small p-value** (below 0.05) lets us
  *reject* the null, i.e. the series *looks stationary*.

That 0.05 comparison is the one line hidden inside `is_it_steady`. Here we print
the p-values themselves and watch the price's large value collapse to the
returns' tiny one. `statsmodels` runs the whole test in one call; the p-value is
the value at index 1.

In [ ]:
from statsmodels.tsa.stattools import adfuller

p_value_price = adfuller(close_price.values)[1]
p_value_returns = adfuller(log_returns.values)[1]

print("ADF p-value for the PRICE  :", round(p_value_price, 4),
      "-> large p-value: non-stationary (this is why the verdict was 'no')")
print("ADF p-value for the RETURNS:", round(p_value_returns, 6),
      "-> tiny p-value: stationary (this is why the verdict was 'yes')")
print()
print("The 0.05 rule below is exactly what is_it_steady checks for you:")
print("  price   < 0.05 ?", p_value_price < 0.05)
print("  returns < 0.05 ?", p_value_returns < 0.05)

### Stretch (optional) — the ACF: does the past predict the future?

Skip this if you're new. The **ACF** (autocorrelation function) measures how
related the series is to itself *k* days ago, for each lag *k*. Think of it as
asking: "does today's return tell me anything about the return a few days
later?" The dashed band is the 'significance' region; bars poking outside it are
the meaningful ones.

For returns, the ACF is almost all inside the band, meaning today's return
barely depends on past returns. In plain words: you **cannot predict tomorrow's
direction** from the past. That is roughly the 'efficient market' idea.

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

# plot_acf draws the autocorrelation bars for us. We look at 25 lags (days).
figure, axis = plt.subplots(figsize=(9, 4.5))
plot_acf(log_returns.values, lags=25, ax=axis)
axis.set_title("ACF of returns: almost no correlation with the past")
axis.set_xlabel("lag (days)")
plt.show()

### Stretch (optional) — the PACF: direct correlation at each lag

Skip this if you're new. The **PACF** (partial autocorrelation function) is the
same idea, but at each lag it strips out the effect of the shorter lags in
between. It is the 'direct' correlation at lag *k*, with the middle days'
influence removed.

Together the ACF and PACF are the **fingerprints** used to choose a time-series
model. We don't pick a model today; we just learn to *read* these plots. Next
session (ARIMA) we use the fingerprints to choose the model's settings.

In [ ]:
from statsmodels.graphics.tsaplots import plot_pacf

# plot_pacf draws the partial-autocorrelation bars. method="ywm" is a standard,
# well-behaved choice for the calculation.
figure, axis = plt.subplots(figsize=(9, 4.5))
plot_pacf(log_returns.values, lags=25, ax=axis, method="ywm")
axis.set_title("PACF of returns")
axis.set_xlabel("lag (days)")
plt.show()

### Stretch (optional) — the memory hides in the *size* of the moves

Skip if you're new. Here is the subtle point behind volatility clustering. The
returns themselves have almost no memory (their ACF is flat, above). But the
*absolute* returns, the sizes of the moves ignoring direction, do have memory: a
big move today makes a big move tomorrow more likely. We plot the ACF of
`|returns|` and watch the bars stay well outside the band for many lags. That
lasting correlation is volatility clustering, made precise.

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

# The size of each move, ignoring its direction.
absolute_returns = np.abs(log_returns.values)

figure, axis = plt.subplots(figsize=(9, 4.5))
plot_acf(absolute_returns, lags=25, ax=axis)
axis.set_title("ACF of |returns|: the size of moves has lasting memory")
axis.set_xlabel("lag (days)")
plt.show()

## What you just did

You saw the split that the whole finance block rests on: a **price wanders off**
while its **returns sit still**. You confirmed it with one plain check,
`is_it_steady`, and watched the verdict flip from **no** for the price to
**yes** for its returns. That is the data foundation for the module: pull data,
convert to returns, check it is steady, then model it.

If you went through the Stretch cells, you also met the ADF hypothesis test
inside the check and the ACF and PACF plots. Next session we turn the steady
return series into real forecasts, using exactly those ACF and PACF
fingerprints.